In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
import html

## Read and Preprocess Data

In [2]:
# read raw data
prices = pd.read_csv("top40_marketdata_open_close_july_dec_2025.csv")
news = pd.read_csv("top40_headlines_july_dec_2025.csv")

In [3]:
# We need to (1) convert UTC time to Eastern Time
# (2) Then add returns sign on the news data
TZ = "America/New_York"
MORNING_CUTOFF = "09:00"   
CLOSE_TIME = "16:00"      

def hhmm_to_minutes(hhmm: str) -> int:
    h, m = map(int, hhmm.split(":"))
    return 60*h + m

morning_min = hhmm_to_minutes(MORNING_CUTOFF)
close_min   = hhmm_to_minutes(CLOSE_TIME)

# Add overnight signs / returns
prices = prices.copy()
prices["rp_entity_id"] = prices["rp_entity_id"].astype(str)
prices["marketdate"] = pd.to_datetime(prices["marketdate"], errors="coerce").dt.normalize()
prices = prices.dropna(subset=["marketdate"]).copy()
prices = prices.sort_values(["rp_entity_id", "marketdate"], kind="mergesort").reset_index(drop=True)

prices["close_prev"] = prices.groupby("rp_entity_id")["close"].shift(1)
prices["overnight_ret"] = prices["open"] / prices["close_prev"] - 1

prices["overnight_sign"] = pd.Series(
    np.where(prices["overnight_ret"].notna(),
             (prices["overnight_ret"] > 0).astype(int),
             pd.NA),
    dtype="Int64"
)

labels = prices[["rp_entity_id", "marketdate", "overnight_ret", "overnight_sign"]].copy()

cal = (prices[["rp_entity_id", "marketdate"]]
       .drop_duplicates()
       .dropna(subset=["marketdate"])
       .sort_values(["rp_entity_id", "marketdate"], kind="mergesort")
       .reset_index(drop=True))


# Keep overnight news
news = news.copy()
news["rp_entity_id"] = news["rp_entity_id"].astype(str)

news["timestamp_utc"] = pd.to_datetime(news["timestamp_utc"], utc=True, errors="coerce")
news = news.dropna(subset=["timestamp_utc"]).copy()

news["timestamp_et"] = news["timestamp_utc"].dt.tz_convert(TZ)

tod_min = (news["timestamp_et"].dt.hour * 60 +
           news["timestamp_et"].dt.minute +
           news["timestamp_et"].dt.second / 60.0)

news["is_pre"]   = tod_min < morning_min
news["is_after"] = tod_min >= close_min
news_overnight = news[news["is_pre"] | news["is_after"]].copy()


# Candidate date in ET: 
# - pre-market (before cutoff) -> same calendar day
# - after-close -> next calendar day
# Note: candidate date is a naive date we try to predict. In practice, it may not be a trading date. The next chunk computes the trade_date!!
news_overnight["date_et"] = news_overnight["timestamp_et"].dt.normalize().dt.tz_localize(None)
news_overnight["candidate_date"] = news_overnight["date_et"] + pd.to_timedelta(
    news_overnight["is_after"].astype(int), unit="D"
)

news_overnight["candidate_date"] = pd.to_datetime(news_overnight["candidate_date"], errors="coerce")
news_overnight = news_overnight.dropna(subset=["candidate_date"]).copy()



def map_trade_date(one_firm_news: pd.DataFrame) -> pd.DataFrame:
    """Map each news row -> next trading day (group-wise merge_asof)"""
    rid = one_firm_news["rp_entity_id"].iloc[0]
    firm_cal = cal[cal["rp_entity_id"] == rid][["marketdate"]].copy()

    # If we somehow have no calendar for this firm, return NA trade_date
    if firm_cal.empty:
        out = one_firm_news.copy()
        out["trade_date"] = pd.NaT
        return out

    one_firm_news = one_firm_news.sort_values("candidate_date", kind="mergesort").reset_index(drop=True)
    firm_cal = firm_cal.sort_values("marketdate", kind="mergesort").reset_index(drop=True)

    mapped = pd.merge_asof(
        one_firm_news,
        firm_cal,
        left_on="candidate_date",
        right_on="marketdate",
        direction="forward",
        allow_exact_matches=True
    ).rename(columns={"marketdate": "trade_date"})

    return mapped


news_mapped = (news_overnight
               .sort_values(["rp_entity_id", "candidate_date"], kind="mergesort")
               .groupby("rp_entity_id", group_keys=False)
               .apply(map_trade_date)
               .reset_index(drop=True))


news_labeled = (news_mapped
    .merge(labels,
           left_on=["rp_entity_id", "trade_date"],
           right_on=["rp_entity_id", "marketdate"],
           how="left")
    .drop(columns=["marketdate"], errors="ignore")
)

# filter out na sign: they are on the boundary dates :
news_labeled = news_labeled[news_labeled["overnight_sign"].notna()]

# delete duplicate headlines on firm date level
news_labeled = (
    news_labeled
      .sort_values(["rp_entity_id", "trade_date", "timestamp_utc"], kind="mergesort")
      .drop_duplicates(subset=["rp_entity_id", "trade_date", "headline"], keep="first")
      .reset_index(drop=True)
      )

/var/folders/fl/b_lsl5qs0vx0yk493rcm5_480000gn/T/ipykernel_17454/720909411.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(map_trade_date)


In [4]:
# clean head line data

_SUFFIX_SOURCE_RE = re.compile(
    r"""
    \s*
    (?:--|\|)\s*                     # delimiter: -- or |
    (                                # source token
      [A-Za-z][A-Za-z0-9.&'-]{1,40}  # e.g., WSJ, Barrons.com
      (?:\.[A-Za-z]{2,10})?          # optional .com / .net / etc.
    )
    \s*$
    """,
    re.VERBOSE
)

_URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
_WS_RE = re.compile(r"\s+")
_ZERO_WIDTH_RE = re.compile(r"[\u200B-\u200D\uFEFF]")
_TRAILING_ELLIPSIS_RE = re.compile(r"(?:\.\.\.|…)\s*$")

def clean_headline(text, lowercase: bool = True) -> str:
    if pd.isna(text):
        return ""

    s = str(text)
    s = html.unescape(s)
    s = unicodedata.normalize("NFKC", s)

    s = _URL_RE.sub(" ", s)
    s = _ZERO_WIDTH_RE.sub("", s)

    # strip source suffix first, then ellipsis
    s = _SUFFIX_SOURCE_RE.sub("", s).strip()
    s = _TRAILING_ELLIPSIS_RE.sub("", s).rstrip()

    if lowercase:
        s = s.lower()

    s = _WS_RE.sub(" ", s).strip()
    return s

news_labeled["headline_clean"] = news_labeled["headline"].map(clean_headline)

In [5]:
r_threshold = 50
news_labeled = news_labeled[news_labeled["relevance"]>=r_threshold]

In [6]:
# aggregate on firm-date level
def make_firm_date_table(
    df: pd.DataFrame,
    firm_col: str = "rp_entity_id",
    date_col: str = "trade_date",
    y_col: str = "overnight_sign",
    source_col: str = "source_name",
    timestamp_col = "timestamp_et"
):
    key = [firm_col, date_col]
    df_clean = df.copy()
    stats = (
        df_clean.groupby(key, sort=False)
        .agg(
            overnight_sign=(y_col, "first"),
            n_headlines_unique=("headline_clean", "size"),
            n_sources=("source_name", "nunique"),
            avg_relevance = ("relevance", "mean"),
            pr_proportion = ("is_press_release", "mean")
        )
        .reset_index()
    )
    df_for_agg = df_clean
    subset = key + ["headline_clean"]   
    df_for_agg = df_clean.sort_values(timestamp_col).drop_duplicates(subset=subset, keep="first")

    # Aggregate lists
    agg_dict = {"headlines": ("headline_clean", list), "sources": (source_col, list)}
    agg_dict["timestamps"] = (timestamp_col, list)
    df_lists = df_for_agg.groupby(key, sort=False).agg(**agg_dict).reset_index()
    #df_lists["n_headlines"] = df_lists["headlines"].map(len)
    df_firm_date = stats.merge(df_lists, on=key, how="left")
    return df_firm_date


df_firm_date = make_firm_date_table(news_labeled)

In [7]:
# train_df = df_firm_date[df_firm_date['trade_date'].dt.month.isin([9,10])]
# train_df.to_csv("firm_date_train.csv", index=False)
# val_df = df_firm_date[df_firm_date['trade_date'].dt.month == 11]
# val_df.to_csv("firm_date_val.csv", index=False)
# test_df = df_firm_date[df_firm_date['trade_date'].dt.month == 12]
# test_df.to_csv("firm_date_test.csv", index=False)
# news_labeled.to_csv("top40_headlines_sep_dec_2025_cleaned.csv", index=False)

In [8]:
train_df = df_firm_date[df_firm_date['trade_date'].dt.month.isin([7,8])]
train_df.to_csv("firm_date_train_r50.csv", index=False)
val_df = df_firm_date[df_firm_date['trade_date'].dt.month.isin([9,10])]
val_df.to_csv("firm_date_val_r50.csv", index=False)
test_df = df_firm_date[df_firm_date['trade_date'].dt.month.isin([11,12])]
test_df.to_csv("firm_date_test_r50.csv", index=False)
news_labeled.to_csv("top40_headlines_july_dec_2025_cleaned_r50.csv", index=False)

In [9]:
train_df.shape

(1253, 10)

In [10]:
val_df.shape

(1330, 10)

In [11]:
test_df.shape

(1252, 10)

In [12]:
news_labeled.shape

(38757, 29)